<a href="https://colab.research.google.com/github/niveditha-bh/WorkflowLLM/blob/main/workflow_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers peft bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import json

with open('/content/drive/MyDrive/dataset/train_10pct.json') as f:
    subset = json.load(f)

print("Loaded examples:", len(subset))

Loaded examples: 10556


In [5]:
def format_example(ex):
    query = ex.get("query", "")
    workflow_code = ex.get("workflow_code", "")
    task_plan = ex.get("task_plan", "")

    if task_plan:
        text = f"### Task:\n{query}\n\n### Plan:\n{task_plan}\n\n### Workflow Code:\n{workflow_code}"
    else:
        text = f"### Task:\n{query}\n\n### Workflow Code:\n{workflow_code}"

    return text

texts = [format_example(ex) for ex in subset]
print("Formatted examples:", len(texts))

Formatted examples: 10556


In [6]:
from huggingface_hub import login
login()  # paste your approved HF token when prompted

In [7]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "meta-llama/Llama-3.1-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

dataset = Dataset.from_dict({"text": texts})

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=1536, padding="max_length")

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])
print(tokenized_dataset)

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Map:   0%|          | 0/10556 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 10556
})


In [11]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded successfully.")
print("Memory footprint:", model.get_memory_footprint() / 1e9, "GB")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Model loaded successfully.
Memory footprint: 5.591540224 GB


In [12]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import os

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/dataset/workflowllm-lora-10pct",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    max_steps=300,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

last_checkpoint = None
if os.path.isdir(training_args.output_dir):
    checkpoints = [d for d in os.listdir(training_args.output_dir) if d.startswith("checkpoint")]
    if checkpoints:
        last_checkpoint = os.path.join(training_args.output_dir, sorted(checkpoints)[-1])
        print(f"Resuming from: {last_checkpoint}")

trainer.train(resume_from_checkpoint=last_checkpoint)

Step,Training Loss
20,1.182917
